In [ ]:
# Must be run on Derecho.

import os

OUTPUT_BUCKET = "s3://reflective-data-store/CESM2-WACCM"

# Cloudflare R2 credentials. Region MUST be "auto" for R2.
R2_KEY = os.environ.get("R2_ACCESS_KEY", None)
R2_SECRET = os.environ.get("R2_SECRET_ACCESS_KEY", None)
R2_ENDPOINT = f"https://{os.environ.get("R2_ACCOUNT_ID", None)}.r2.cloudflarestorage.com"
R2_STORAGE_OPTIONS = {
    "key":           R2_KEY,
    "secret":        R2_SECRET,
    "endpoint_url":  R2_ENDPOINT,
    "client_kwargs": {"region_name": "auto"},
}

# One zarr store per scenario
STORES = {
    "G6-1.5K-HiLLA": f"{OUTPUT_BUCKET}/CESM2-WACCM_G6-1.5K-HiLLA.zarr",
    "G6-1.5K-SAI":   f"{OUTPUT_BUCKET}/CESM2-WACCM_G6-1.5K-SAI.zarr",
    "SSP2-4.5":      f"{OUTPUT_BUCKET}/CESM2-WACCM_SSP2-4.5.zarr",
}

# Per-scenario root path, case stem, and ensemble members.
EXPERIMENTS = {
    "G6-1.5K-HiLLA": {
        "base_url":  "/glade/campaign/cgd/amp/walkerl/GeoMIP/HiLLA",
        "case_stem": "b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA",
        "members":   ["001", "002", "003"],
    },
    "G6-1.5K-SAI": {
        "base_url":  "/glade/campaign/cesm/collections/ARISE-SAI-1.5",
        "case_stem": "b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI",
        "members":   ["001", "002", "003"],
    },
    "SSP2-4.5": {
        "base_url":  "/glade/campaign/cesm/collections/CESM2-WACCM-SSP245",
        "case_stem": "b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM",
        "members":   [f"{i:03d}" for i in range(1, 11)],
    },
}
ALL_EXPERIMENTS = list(EXPERIMENTS.keys())


# Variable lists by CMIP table + realm
# Essential + High, direct 1:1 CESM name only.

AMON_2D_VARS = [
    # Essential
    "TREFHT", "TREFHTMX", "TREFHTMN", "TS", "PRECT", "PRECC",
    "PS", "PSL", "U10", "RHREFHT", "QREFHT", "CLDTOT", "ICEFRAC",
    "FSUTOA", "FSDS", "FSNTOAC", "FSDSC",
    "FLUT", "FLUTC", "FLDS", "FLDSC",
    "AODVISdn", "BURDENSO4dn",
    # High
    "ACTREL", "DF_SO2", "DF_H2SO4", "WD_SO2", "WD_H2SO4",
    "SFSO2", "AODSO4dn",
    "TGCLDIWP", "TGCLDCWP", "TMQ",
    "TAUX", "TAUY", "LHFLX", "SHFLX",
]
AMON_3D_VARS = [
    # Essential
    "T", "U", "V", "Z3",
    # High
    "RELHUM", "Q", "OMEGA", "O3",
    "REFF_AERO", "ABSORB", "CLOUD","CLDICE", "CLDLIQ"
]

# atm / day_1 / cam.h1  → CMIP day
DAY_2D_VARS = [
    # Essential
    "TREFHT", "TREFHTMX", "TREFHTMN", "PRECT", "RHREFHT",
    # High
    "QREFHT", "PSL", "FSDS", "FLDS", "FSDSC", "FLDSC",
    "CLDTOT", "U10", "TS", "PS",
]
# CMIP siconca at daily cadence comes from CICE's daily tape (cice.h1) as
# `aice_d`, not from cam.h1.ICEFRAC — CESM rarely outputs ICEFRAC at daily.
SIDAY_2D_VARS = ["aice_d"]

# ocn / month_1 / pop.h  → Omon. (sos = SALT[k=1] is a slice, not a standalone
# variable → skip; downstream can slice from 3D SALT if needed.)
OMON_2D_VARS = [
    # Essential
    "SST", "SSH", "HMXL_DR", "SHF", "TAUX", "TAUY",
    # High
    "FG_CO2", "DpCO2",
]
OMON_3D_VARS = [
    # High
    "TEMP", "SALT", "UVEL", "VVEL", "WVEL",
]

# ocn / year_1 / pop.h.ecosys.nyear1  → Oyr
OYR_3D_VARS = ["O2"]

# ice / month_1 / cice.h  → SImon. CICE filenames are lowercase.
SIMON_2D_VARS = ["aice", "hi", "hs", "uvel", "vvel", "Tsfc"]

# lnd / month_1 / clm2.h0  → Lmon
LMON_2D_VARS = [
    "SOILWATER_10CM", "QOVER", "QRUNOFF", "QSOIL",
    "TLAI", "GPP", "NPP",
]

# lnd / day_1 / clm2.h2  → CMIP day (land)
LDAY_2D_VARS = ["FSNO", "TSOI"]


# Each group carries the routing tuple (component, freq_dir, h_tape, comp_name)
# so open_variable can build the full path directly from scenario + group.
#
# Grid / level assumptions:
#   atm (cam):  lat=192, lon=288, lev=70  (CESM2-WACCM)
#   ocn (pop):  nlat=384, nlon=320, z_t=60  (gx1v7)
#   ice (cice): nj=384, ni=320
#   lnd (clm2): lat=192, lon=288
# Confirm by opening one file once paths resolve.

tables = {
    "Amon": {
        "experiments": ALL_EXPERIMENTS,
        "groups": {
            "atmos_2d": {
                "component": "atm", "freq_dir": "month_1",
                "comp_name": "cam", "h_tape": "h0",
                "variables": AMON_2D_VARS,
                "chunks": {"member": 1, "time": 120, "lat": 192, "lon": 288},
            },
            "atmos_3d": {
                "component": "atm", "freq_dir": "month_1",
                "comp_name": "cam", "h_tape": "h0",
                "variables": AMON_3D_VARS,
                "chunks": {"member": 1, "time": 60, "lev": 10, "lat": 192, "lon": 288},
            },
        },
    },
    "day": {
        "experiments": ALL_EXPERIMENTS,
        "groups": {
            "atmos_2d": {
                "component": "atm", "freq_dir": "day_1",
                "comp_name": "cam", "h_tape": "h1",
                "variables": DAY_2D_VARS,
                "chunks": {"member": 1, "time": 365, "lat": 192, "lon": 288},
            },
        },
    },
    "Omon": {
        "experiments": ALL_EXPERIMENTS,
        "groups": {
            "ocean_2d": {
                "component": "ocn", "freq_dir": "month_1",
                "comp_name": "pop", "h_tape": "h",
                "variables": OMON_2D_VARS,
                "chunks": {"member": 1, "time": 120, "nlat": 384, "nlon": 320},
            },
            "ocean_3d": {
                "component": "ocn", "freq_dir": "month_1",
                "comp_name": "pop", "h_tape": "h",
                "variables": OMON_3D_VARS,
                "chunks": {"member": 1, "time": 12, "z_t": 10, "nlat": 384, "nlon": 320},
            },
        },
    },
    "Oyr": {
        "experiments": ALL_EXPERIMENTS,
        "groups": {
            "ocean_bgc_3d": {
                "component": "ocn", "freq_dir": "year_1",
                "comp_name": "pop", "h_tape": "h.ecosys.nyear1",
                "variables": OYR_3D_VARS,
                "chunks": {"member": 1, "time": 10, "z_t": 10, "nlat": 384, "nlon": 320},
            },
        },
    },
    "SImon": {
        "experiments": ALL_EXPERIMENTS,
        "groups": {
            "seaice_2d": {
                "component": "ice", "freq_dir": "month_1",
                "comp_name": "cice", "h_tape": "h",
                "variables": SIMON_2D_VARS,
                "chunks": {"member": 1, "time": 120, "nj": 384, "ni": 320},
            },
        },
    },
    "SIday": {
        "experiments": ALL_EXPERIMENTS,
        "groups": {
            "seaice_2d": {
                "component": "ice", "freq_dir": "day_1",
                "comp_name": "cice", "h_tape": "h1",
                "variables": SIDAY_2D_VARS,
                "chunks": {"member": 1, "time": 365, "nj": 384, "ni": 320},
            },
        },
    },
    "Lmon": {
        "experiments": ALL_EXPERIMENTS,
        "groups": {
            "land_2d": {
                "component": "lnd", "freq_dir": "month_1",
                "comp_name": "clm2", "h_tape": "h0",
                "variables": LMON_2D_VARS,
                "chunks": {"member": 1, "time": 120, "lat": 192, "lon": 288},
            },
        },
    },
    "Lday": {
        "experiments": ALL_EXPERIMENTS,
        "groups": {
            "land_2d": {
                "component": "lnd", "freq_dir": "day_1",
                "comp_name": "clm2", "h_tape": "h2",
                "variables": LDAY_2D_VARS,
                "chunks": {"member": 1, "time": 365, "lat": 192, "lon": 288},
            },
        },
    },
}


In [ ]:
import glob
import json
import gc
import os

import dask
import numpy as np
import s3fs
import xarray as xr
import zarr

# ─ s3fs multipart workaround
# zarr.FsspecStore.set -> s3fs._pipe_file flips to multipart upload whenever a
# value exceeds `chunksize` (default 50 MB). Our zarr chunks for atm 3D, day
# 2D, ocean 2D/3D, SIday, etc. all exceed that threshold (largest ~179 MB).
# Under R2 with adaptive boto retries, an in-flight part can be resubmitted —
# R2 keeps the latest ETag for that part number, the local s3fs `parts` list
# still holds the earlier ETag, and CompleteMultipartUpload fails with
# `InvalidPart: One or more of the specified parts could not be found.`
# Sidestep the entire failure mode by raising the threshold past our largest
# chunk; R2 accepts single PutObject up to 5 GB.
#
# Idempotent: stash the genuine original on a class attribute the first time
# this cell runs, then always delegate to that. Without this, re-running the
# cell layers a wrapper on top of the wrapper and the next call recurses
# forever (RecursionError).
import s3fs.core as _s3fs_core

if not hasattr(_s3fs_core.S3FileSystem, "_pipe_file_unpatched"):
    _s3fs_core.S3FileSystem._pipe_file_unpatched = _s3fs_core.S3FileSystem._pipe_file


async def _pipe_file_single_put(
    self, path, value, chunksize=1024 * 1024 * 1024, max_concurrency=None, **kwargs
):
    return await _s3fs_core.S3FileSystem._pipe_file_unpatched(
        self, path, value,
        chunksize=chunksize,
        max_concurrency=max_concurrency,
        **kwargs,
    )


_s3fs_core.S3FileSystem._pipe_file = _pipe_file_single_put
from zarr.storage import FsspecStore

# R2-backed filesystem. R2_STORAGE_OPTIONS is defined in the config cell.
# `config_kwargs` extends botocore client config with longer per-request
# timeouts and an adaptive retry policy so transient R2 stalls (we observed
# AioReadTimeoutError on chunk GETs partway through a write) don't abort
# the whole pipeline.
_R2_BOTO_CONFIG = {
    "connect_timeout": 60,
    "read_timeout": 300,
    "retries": {"max_attempts": 10, "mode": "adaptive"},
}
fs = s3fs.S3FileSystem(config_kwargs=_R2_BOTO_CONFIG, **R2_STORAGE_OPTIONS)

from dask.distributed import Client

import warnings
import logging

warnings.filterwarnings("ignore", message=".*vlen-utf8.*")
logging.getLogger("distributed.shuffle._scheduler_plugin").setLevel(logging.ERROR)

client = Client(n_workers=4, threads_per_worker=2, memory_limit="6GB")
print(client.dashboard_link)

# Helpers

# CESM freq_dir -> time chunk applied on read. Aligning with the intended zarr
# output chunking avoids a global shuffle at write time.
_FREQ_TIME_CHUNK = {
    "month_1": 120,
    "day_1":   365,
    "day_5":   365,
    "hour_1":  8760,
    "hour_3":  2920,
    "hour_6":  1460,
    "year_1":  50,
}


def open_variable(experiment, group_config, var, member):
    """Open NetCDF file(s) for (experiment, var, member) using group routing.

    File layout:
      {base_url}/{case_stem}.{member}/{component}/proc/tseries/{freq_dir}/
        {case_stem}.{member}.{comp_name}.{h_tape}.{var}.{time}.nc
    Multiple time-chunk files per variable are concatenated via open_mfdataset.
    """
    exp = EXPERIMENTS[experiment]
    member_dir = f"{exp['case_stem']}.{member}"
    sub = f"{group_config['component']}/proc/tseries/{group_config['freq_dir']}"
    fname_prefix = (
        f"{exp['case_stem']}.{member}."
        f"{group_config['comp_name']}.{group_config['h_tape']}.{var}"
    )
    pattern = f"{exp['base_url']}/{member_dir}/{sub}/{fname_prefix}.*.nc"
    paths = sorted(glob.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"No files matching: {pattern}")

    open_chunks = {"time": _FREQ_TIME_CHUNK.get(group_config["freq_dir"], 120)}
    # HiLLA daily (h1) time-series files overlap at decade boundaries
    # (e.g. 20391231-20500105.nc shares day 2039-12-31 with 20350101-20391231.nc,
    # and 20500101-20600105.nc overlaps the previous by 5 days). combine_by_coords
    # rejects non-monotonic time indexes, so use combine="nested" to just stack
    # the files, then sort + drop duplicate timestamps ourselves. This is a
    # no-op for clean monthly data.
    ds = xr.open_mfdataset(
        paths,
        engine="netcdf4",
        chunks=open_chunks,
        combine="nested",
        concat_dim="time",
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )
    ds = ds.sortby("time").drop_duplicates("time", keep="first")

    # POP decodes a noleap / 365_day calendar to cftime, which trips
    # xarray.namedarray.parallelcompat.ChunkManagerEntrypoint.rechunk
    # downstream: its cftime branch calls _get_chunk without dims, and
    # zip(dims=(), shape, strict=True) raises. Convert to datetime64[ns]
    # while the date range fits (the runs here sit in the 1900-2100 window).
    if "time" in ds.indexes and isinstance(ds.indexes["time"], xr.CFTimeIndex):
        ds = ds.assign_coords(
            time=ds.indexes["time"].to_datetimeindex(time_unit="ns")
        )
    # POP emits `time_bound` (no `s`) as a 2D cftime array. It escapes the
    # `endswith("_bnds")` filter downstream and keeps the rechunk cftime path
    # active even after the time-coord conversion above. Drop any ancillary
    # object-dtype data var here so the next `.chunk(...)` call is clean.
    aux_drops = [
        v for v in ds.data_vars
        if v in ("time_bound", "time_bounds") or ds[v].dtype == object
    ]
    if aux_drops:
        ds = ds.drop_vars(aux_drops)

    data_vars = [v for v in ds.data_vars if not v.endswith("_bnds")]
    if len(data_vars) == 1 and data_vars[0] != var:
        ds = ds.rename({data_vars[0]: var})
    return ds


def _patch_array_dims(fs, bucket_path, rel_path, dims):
    """Ensure dimension metadata is set for an array, regardless of whether it
    was serialized in zarr v3 format (zarr.json) or v2 format (.zarray +
    .zattrs). The env here runs zarr-python v2 writing v3 format, and v2's
    require_dataset with `object_codec=VLenUTF8()` can fall back to v2 array
    metadata inside a v3 group — so we handle both.

    For v3: set `dimension_names` on the root and `_ARRAY_DIMENSIONS` inside
    `attributes`.
    For v2: set `_ARRAY_DIMENSIONS` on `.zattrs` (creating it if absent).
    """
    base = f"{bucket_path}/{rel_path}"
    fs.invalidate_cache(base)
    v3_path = f"{base}/zarr.json"
    v2_array = f"{base}/.zarray"
    v2_attrs = f"{base}/.zattrs"

    if fs.exists(v3_path):
        with fs.open(v3_path, "r") as f:
            meta = json.load(f)
        meta["dimension_names"] = dims
        meta.setdefault("attributes", {})["_ARRAY_DIMENSIONS"] = dims
        with fs.open(v3_path, "w") as f:
            json.dump(meta, f)
        return

    if fs.exists(v2_array):
        attrs = {}
        if fs.exists(v2_attrs):
            with fs.open(v2_attrs, "r") as f:
                attrs = json.load(f)
        attrs["_ARRAY_DIMENSIONS"] = dims
        with fs.open(v2_attrs, "w") as f:
            json.dump(attrs, f)
        return

    # Neither format present — surface what IS in the directory so the next
    # iteration has a real clue to work with.
    try:
        contents = fs.ls(base)
    except Exception as exc:
        contents = f"(ls failed: {exc!r})"
    raise FileNotFoundError(
        f"No array metadata at {base}: neither zarr.json (v3) nor "
        f".zarray (v2) exists. Directory contents: {contents}"
    )


# Legacy alias — keep existing callers functional while we migrate.
_patch_zarr_json = _patch_array_dims


def _coord_meta_exists(fs, bucket_path, coord_path):
    """True if the coord array has either v3 (zarr.json) or v2 (.zarray)
    metadata on disk."""
    base = f"{bucket_path}/{coord_path}"
    fs.invalidate_cache(base)
    return fs.exists(f"{base}/zarr.json") or fs.exists(f"{base}/.zarray")


def _ensure_coord_array(fs, root, bucket_path, zarr_group_path, coord_name, coord_da):
    """Ensure a coord array exists in the zarr group. xarray.to_zarr can skip
    string-dtype dim coords when zarr-python v2 is writing v3 format (the
    `member` coord is the usual offender here).

    Strategy:
      1. Try require_dataset with VLenUTF8 object codec (writes either v3 or
         v2 metadata depending on what zarr-v2 is willing to emit).
      2. If that still leaves the array unpersisted, coerce to fixed-width
         unicode and retry — numpy `<U` dtype avoids the codec entirely.

    Whichever strategy succeeds, the caller's _patch_array_dims step will find
    the metadata file (v3 or v2) and set the dim attributes correctly.
    """
    coord_path = f"{zarr_group_path}/{coord_name}"
    if _coord_meta_exists(fs, bucket_path, coord_path):
        return

    values = np.asarray(coord_da.values)
    dims = list(coord_da.dims)
    chunks = values.shape if values.shape else (1,)
    is_string = values.dtype.kind in ("O", "U")

    if is_string:
        print(f"    (fallback) writing string coord {coord_name} via require_dataset + VLenUTF8")
        from numcodecs import VLenUTF8
        str_values = np.array(
            [str(v) for v in values.flat], dtype=object
        ).reshape(values.shape)
        arr = root.require_dataset(
            coord_name,
            shape=str_values.shape,
            chunks=chunks,
            dtype=object,
            object_codec=VLenUTF8(),
            dimension_names=dims,
        )
        arr[:] = str_values
    else:
        arr = root.require_dataset(
            coord_name,
            shape=values.shape,
            chunks=chunks,
            dtype=values.dtype,
            dimension_names=dims,
        )
        arr[:] = values

    if _coord_meta_exists(fs, bucket_path, coord_path):
        return

    # First attempt didn't land on disk; try fixed-width unicode (skips the
    # object codec path entirely — this is what v3 can always emit).
    if is_string:
        print(f"    (fallback-2) retrying coord {coord_name} as fixed-width unicode")
        max_len = max((len(str(v)) for v in values.flat), default=1)
        u_values = np.array(
            [str(v) for v in values.flat], dtype=f"<U{max_len}"
        ).reshape(values.shape)
        arr = root.require_dataset(
            coord_name,
            shape=u_values.shape,
            chunks=chunks,
            dtype=u_values.dtype,
            dimension_names=dims,
        )
        arr[:] = u_values

    if not _coord_meta_exists(fs, bucket_path, coord_path):
        try:
            listing = fs.ls(f"{bucket_path}/{coord_path}")
        except Exception as exc:
            listing = f"(ls failed: {exc!r})"
        raise RuntimeError(
            f"Unable to persist coord {coord_name!r} via either write "
            f"strategy. Directory state: {listing}"
        )


def _write_var_to_zarr(combined, var, store_url, zarr_group_path, chunks, first_write):
    """Write one variable to a zarr group. Coordinates go through xarray on the
    first write; data variables go through raw zarr so we keep full control of
    dimension metadata."""
    group_url = f"{store_url}/{zarr_group_path}"
    bucket_path = store_url.replace("s3://", "")
    storage_options = R2_STORAGE_OPTIONS

    if first_write:
        coords_only = combined.drop_vars(var)
        coords_only.to_zarr(
            group_url,
            mode="w",
            consolidated=False,
            storage_options=storage_options,
        )
        # s3fs may still hold a "does not exist" cache entry under this prefix
        # from before to_zarr wrote; flush so the patch loop can see the fresh
        # coord arrays.
        fs.invalidate_cache(f"{bucket_path}/{zarr_group_path}")

        # xarray.to_zarr occasionally skips coord arrays under zarr v3 (string
        # dtypes in particular — `member` is the usual offender on R2). Make
        # sure each coord has an array; write it via raw zarr if missing.
        root = zarr.open_group(
            group_url, mode="a", storage_options=storage_options
        )
        for coord_name in coords_only.coords:
            coord_da = coords_only[coord_name]
            _ensure_coord_array(
                fs, root, bucket_path, zarr_group_path,
                coord_name, coord_da,
            )
            # Use the coord's true dims — auxiliary 2D coords like POP's
            # TLAT/TLONG/area have dims ("nlat", "nlon"), and overwriting
            # dimension_names with [coord_name] (1 element) corrupts the
            # zarr.json so consolidate_metadata fails validation.
            _patch_array_dims(
                fs, bucket_path, f"{zarr_group_path}/{coord_name}",
                list(coord_da.dims),
            )

    root = zarr.open_group(group_url, mode="a", storage_options=storage_options)
    da = combined[var]
    dims = list(da.dims)
    chunk_tuple = tuple(chunks.get(d, s) for d, s in zip(dims, da.shape))

    # Force dask's block shape to match zarr's chunk_tuple exactly. Dimensions
    # that aren't in `chunks` (e.g. POP's `z_w_top` for WVEL when only `z_t`
    # is listed) otherwise keep their open-time block size, leaving dask with
    # multiple blocks where zarr has one full chunk. Zarr then falls into a
    # per-chunk read-modify-write merge — that fires `_read_key` against R2
    # for every dask block, and on a fresh array those GETs sometimes stall
    # for the full read_timeout window. Aligning here makes every store a
    # complete-chunk write so the read path is never invoked.
    da = da.chunk(dict(zip(dims, chunk_tuple)))

    # On the first write to a group, evict any pre-existing copy of this var
    # so the array starts empty. Otherwise dask's store path triggers a
    # read-modify-write merge per chunk against R2, which is both slow and
    # vulnerable to transient stalls (one timed-out GET aborts the whole job).
    if first_write and var in root:
        try:
            del root[var]
            fs.invalidate_cache(f"{bucket_path}/{zarr_group_path}/{var}")
        except KeyError:
            pass

    if var not in root:
        root.require_dataset(
            var,
            shape=da.shape,
            chunks=chunk_tuple,
            dtype=da.dtype,
            dimension_names=dims,
        )

    da.data.store(root[var], lock=False)
    _patch_zarr_json(fs, bucket_path, f"{zarr_group_path}/{var}", dims)


def ensure_root_markers(store_path, table_names):
    """Write zarr.json group markers at root and each table level so
    consolidate_metadata can walk the tree. xarray only writes markers at the
    leaf-group level, so intermediate levels need to be added explicitly.

    Verifies each marker exists after write and retries once. We have observed
    transient SSL aborts under aiohttp on the R2 connection that swallow a
    write silently — without verification, consolidate_metadata then warns
    "Object at <table> is not recognized as a component of a Zarr hierarchy"
    and consumers can't walk into the table.
    """
    bucket_path = store_path.replace("s3://", "")
    marker = {"zarr_format": 3, "node_type": "group", "attributes": {}}
    for p in [""] + table_names:
        key = f"{bucket_path}/{p}/zarr.json" if p else f"{bucket_path}/zarr.json"
        for attempt in (1, 2):
            with fs.open(key, "w") as f:
                json.dump(marker, f)
            fs.invalidate_cache(key)
            if fs.exists(key):
                break
            print(f"  ⚠ marker missing after attempt {attempt}: {key}")
        else:
            raise RuntimeError(f"failed to persist group marker at {key}")
        print(f"  Wrote marker: {key}")


# Processing

def process_group(store_path, experiment, table_name, group_name, group_config, members):
    zarr_group_path = f"{table_name}/{group_name}"
    variables = group_config["variables"]
    chunks = group_config["chunks"]

    print(f"  Processing {zarr_group_path} [{experiment}]...")

    first_write = True
    for var in variables:
        print(f"    Variable: {var}")

        try:
            member_datasets = []
            for m in members:
                ds = open_variable(experiment, group_config, var, m)
                drop = [v for v in ds.data_vars if v.endswith("_bnds")]
                ds = ds.drop_vars(drop).expand_dims({"member": [m]})
                member_datasets.append(ds)
        except FileNotFoundError as exc:
            # Variable not output by this scenario on this h-tape — common for
            # CMIP priority vars that CESM only emits on a different tape for
            # some scenarios (e.g. ICEFRAC on cam.h1 vs cice.h1.aice_d).
            print(f"    ⚠ Skipping {var}: {exc}")
            continue

        # Members sometimes differ by a few days/months at the trailing end;
        # truncate to the shortest common length so concat along "member"
        # gets a consistent time axis.
        min_time = min(ds.sizes.get("time", 0) for ds in member_datasets)
        if min_time > 0:
            member_datasets = [
                ds.isel(time=slice(None, min_time)) for ds in member_datasets
            ]

        combined = xr.concat(member_datasets, dim="member").chunk(chunks)
        _write_var_to_zarr(
            combined, var, store_path, zarr_group_path, chunks, first_write
        )
        first_write = False

        del combined, member_datasets
        gc.collect()

    print("    ✓ Written")


# Main loop
print(tables)
for exp, store_path in STORES.items():
    print(f"\n{'='*50}")
    print(f"Writing store: {store_path}")
    print(f"{'='*50}")

    members = EXPERIMENTS[exp]["members"]

    for table_name, table_config in tables.items():
        if exp not in table_config["experiments"]:
            continue

        for group_name, group_config in table_config["groups"].items():
            process_group(
                store_path, exp, table_name, group_name, group_config, members
            )

    ensure_root_markers(store_path, list(tables.keys()))

    # Consolidate via an explicit FsspecStore so the R2 endpoint + creds from
    # R2_STORAGE_OPTIONS are used (plain URL form would fall back to AWS S3).
    # zarr-python v3 runs consolidate_metadata on its own event loop and
    # requires the underlying fsspec fs to have asynchronous=True; passing the
    # sync `fs` we use for manual reads/writes elsewhere produces
    # "Future ... attached to a different loop". Build a one-shot async fs
    # just for this call.
    bucket_path = store_path.replace("s3://", "")
    async_fs = s3fs.S3FileSystem(
        asynchronous=True, config_kwargs=_R2_BOTO_CONFIG, **R2_STORAGE_OPTIONS
    )
    store = FsspecStore(async_fs, path=bucket_path)
    zarr.consolidate_metadata(store)
    print(f"\n✓ Consolidated metadata for {store_path}")

print("\nAll done!")
